# Mestrado em Inteligência Artificial 25/26

# Practical 4 — TicTacToe

Today we will implement the TicTacToe environment and a state feature encoder that we'll use next class to train a REINFORCE agent.

First, let's see what a game looks like.  
**Note:** this cell requires your `TicTacToeEnv` TODOs to be complete.

In [ ]:
from __future__ import annotations

import random
from typing import Callable

TicTacToeState = tuple[int, ...]
TicTacToeAction = int

_WIN_LINES = ((0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6))

def _winner(board: TicTacToeState) -> int:
    for i, j, k in _WIN_LINES:
        s = board[i] + board[j] + board[k]
        if s == 3:
            return 1
        if s == -3:
            return -1
    return 0

class TicTacToeEnv:
    def __init__(self) -> None:
        self.board: TicTacToeState = (0,) * 9
        self.current_player: int = 1

    def reset(self) -> TicTacToeState:
        self.board = (0,) * 9
        self.current_player = 1
        return self.board

    def available_actions(self, state: TicTacToeState) -> list[TicTacToeAction]:
        return [index for index, cell in enumerate(state) if cell == 0]

    def is_terminal(self, state: TicTacToeState) -> bool:
        return _winner(state) != 0 or all(cell != 0 for cell in state)

    def step(self, action: TicTacToeAction) -> tuple[TicTacToeState, float, bool]:
        if action < 0 or action >= len(self.board):
            raise ValueError(f'Invalid action: {action}')
        if self.board[action] != 0:
            raise ValueError(f'Cell {action} is already occupied.')
        board_list = list(self.board)
        board_list[action] = self.current_player
        new_board = tuple(board_list)
        winner = _winner(new_board)
        done = winner != 0 or all(cell != 0 for cell in new_board)
        reward = 1.0 if winner == self.current_player else 0.0
        self.board = new_board
        self.current_player *= -1
        return new_board, reward, done

    def render(self, state: TicTacToeState | None = None) -> None:
        board = self.board if state is None else state
        symbols = {1: 'X', -1: 'O', 0: '.'}
        for row in range(3):
            start = row * 3
            print(' '.join(symbols[board[start + col]] for col in range(3)))
            if row < 2:
                print()

def random_action(env: TicTacToeEnv, state: TicTacToeState) -> TicTacToeAction:
    return random.choice(env.available_actions(state))

Policy = Callable[[TicTacToeEnv, TicTacToeState], TicTacToeAction]

def play_game(
    env: TicTacToeEnv,
    policy_x: Policy,
    policy_o: Policy,
    render: bool = True,
) -> int:
    state = env.reset()
    if render:
        print('Initial board:')
        env.render(state)
        print()

    while not env.is_terminal(state):
        player_label = 'X' if env.current_player == 1 else 'O'
        policy = policy_x if env.current_player == 1 else policy_o
        action = policy(env, state)
        state, reward, done = env.step(action)
        if render:
            print(f'Player {player_label} plays cell {action}:')
            env.render(state)
            print()

    result = _winner(state)
    if render:
        if result == 1:
            print('X wins!')
        elif result == -1:
            print('O wins!')
        else:
            print('Draw!')
    return result

In [3]:
env = TicTacToeEnv()
play_game(env, random_action, random_action)

NotImplementedError: TODO: implement reset.